# 05 Inspect PCA Output Files

这个 notebook 用来快速查看 `X_features_pca.parquet`、`X_mask.parquet`、`Y_phecode.parquet` 的内容、列名、shape，以及确认是否仍然包含原始 `person_id`。  

先运行前几个 cell 即可；如果路径不对，只需要改下面的 `PCA_DIR`。

In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

# 默认路径：如果你是在项目根目录运行，一般就是这个
PCA_DIR = Path('output/pca')

# 如果你要看 split 后的 cohort 文件，可以改成：
# PCA_DIR = Path('output/pca/ad')
# PCA_DIR = Path('output/pca/non_ad')

FILES = {
    'X': PCA_DIR / 'X_features_pca.parquet',
    'mask': PCA_DIR / 'X_mask.parquet',
    'Y': PCA_DIR / 'Y_phecode.parquet',
}

print('Current PCA_DIR:', PCA_DIR.resolve())
for name, path in FILES.items():
    print(f'{name}:', path, 'exists =', path.exists())

## 1. 查看 parquet schema，不完整加载大文件

这个 cell 不会把整个文件读进内存，只看行数、列数、前几十个列名。

In [ ]:
for name, path in FILES.items():
    if not path.exists():
        print(f'[{name}] file not found:', path)
        continue
    pf = pq.ParquetFile(path)
    cols = pf.schema.names
    print('
' + '=' * 80)
    print(f'{name}: {path}')
    print('rows:', pf.metadata.num_rows)
    print('num_columns:', len(cols))
    print('first 40 columns:')
    print(cols[:40])
    print('contains person_id:', 'person_id' in cols)
    print('contains row_id:', 'row_id' in cols)
    print('contains cohort_group:', 'cohort_group' in cols)

## 2. 读取文件并查看前几行

如果文件很大但内存足够，可以直接读。通常 PCA 后的矩阵可以读进来。

In [ ]:
X = pd.read_parquet(FILES['X']) if FILES['X'].exists() else None
mask = pd.read_parquet(FILES['mask']) if FILES['mask'].exists() else None
Y = pd.read_parquet(FILES['Y']) if FILES['Y'].exists() else None

for name, df in {'X': X, 'mask': mask, 'Y': Y}.items():
    if df is None:
        continue
    print(f'{name} shape:', df.shape)
    display(df.head())

## 3. 专门检查 ID 列

这里重点确认输出是否包含原始 `person_id`。

In [ ]:
ID_CANDIDATES = ['row_id', 'person_id', 'cohort_group']

for name, df in {'X': X, 'mask': mask, 'Y': Y}.items():
    if df is None:
        continue
    id_cols = [c for c in ID_CANDIDATES if c in df.columns]
    print('
' + '=' * 80)
    print(f'{name} ID columns:', id_cols)
    print('contains person_id:', 'person_id' in df.columns)
    if id_cols:
        display(df[id_cols].head(10))
    else:
        print('No common ID columns found.')

## 4. 检查 X / mask / Y 是否按 row_id 对齐

如果三个文件都有 `row_id`，这个 cell 会检查 row_id 集合和顺序是否一致。

In [ ]:
dfs = {'X': X, 'mask': mask, 'Y': Y}

for name, df in dfs.items():
    if df is not None and 'row_id' in df.columns:
        print(f'{name}: row_id unique = {df["row_id"].is_unique}, null = {df["row_id"].isna().sum()}')

if all(df is not None and 'row_id' in df.columns for df in dfs.values()):
    print('
Same row count:', len(X) == len(mask) == len(Y))
    print('X vs mask same row_id order:', X['row_id'].equals(mask['row_id']))
    print('X vs Y same row_id order:', X['row_id'].equals(Y['row_id']))
    print('X vs mask same row_id set:', set(X['row_id']) == set(mask['row_id']))
    print('X vs Y same row_id set:', set(X['row_id']) == set(Y['row_id']))

## 5. 查看 PCA feature 列和标签列

这个 cell 会把 ID 列排除掉，分别展示 X 的 PCA embedding 列、mask 列、Y 标签列。

In [ ]:
def non_id_cols(df):
    return [c for c in df.columns if c not in ID_CANDIDATES]

for name, df in {'X': X, 'mask': mask, 'Y': Y}.items():
    if df is None:
        continue
    cols = non_id_cols(df)
    print('
' + '=' * 80)
    print(f'{name}: non-ID columns count = {len(cols)}')
    print(cols[:50])

# 简单看一下 PCA 数值分布
if X is not None:
    x_cols = non_id_cols(X)
    numeric_x_cols = [c for c in x_cols if pd.api.types.is_numeric_dtype(X[c])]
    print('
X numeric PCA columns:', len(numeric_x_cols))
    display(X[numeric_x_cols[:20]].describe().T.head(20))

## 6. 可选：只导出不含 person_id 的检查版文件

默认不会运行。需要的话，把下面 cell 里的 `RUN_EXPORT = False` 改成 `True`。  
注意：这只是导出检查版，不会覆盖原文件。

In [ ]:
RUN_EXPORT = False

if RUN_EXPORT:
    export_dir = PCA_DIR / 'no_person_id_preview'
    export_dir.mkdir(parents=True, exist_ok=True)
    for name, df in {'X_features_pca': X, 'X_mask': mask, 'Y_phecode': Y}.items():
        if df is None:
            continue
        out = df.drop(columns=['person_id'], errors='ignore')
        out_path = export_dir / f'{name}.parquet'
        out.to_parquet(out_path, index=False)
        print('wrote:', out_path, out.shape)
else:
    print('Export skipped. Set RUN_EXPORT = True if needed.')